# 01. Exploratory Data Analysis & Chemical Engineering Thermodynamics

This notebook loads the raw Ashaka Cement Plant kiln datasets (`kiln_energy_dataset.csv` and `kiln_co2_dataset.csv`), merges them explicitly on `SN`, drops the index column to avoid data leakage, and inspects the chemical engineering features defined in Equations 3.13–3.17.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from cement_opt.config import load_config
from cement_opt.data import load_and_merge_datasets, prepare_modeling_dataframe
from cement_opt.features import add_engineering_features

cfg = load_config("../configs/default.yaml")
merged = load_and_merge_datasets(cfg.paths.energy_csv, cfg.paths.co2_csv, join_column=cfg.data.join_column)
enriched = add_engineering_features(
    merged,
    stoichiometric_co2_factor=cfg.thermodynamics.stoichiometric_co2_factor,
    calcination_enthalpy_kj_per_kg=cfg.thermodynamics.calcination_enthalpy_kj_per_kg,
    optimal_o2_percent=cfg.thermodynamics.optimal_o2_percent,
)
print(f"Merged rows: {len(enriched)}, columns: {enriched.shape[1]}")
enriched.head()

In [ ]:
enriched.describe().T[["min", "mean", "50%", "max", "std"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=enriched, x="Fuel", y=cfg.data.co2_target, ax=axes[0])
axes[0].set_title("CO2 Emissions by Fuel Type")
sns.scatterplot(data=enriched, x="Clinker Production Rate (tons/h)", y=cfg.data.co2_target, hue="Fuel", ax=axes[1])
axes[1].set_title("Clinker Production Rate vs CO2 Emissions")
plt.tight_layout()
plt.show()